# Fitting Models

**IMSE 774 · Neural Networks · Week 6 · Sessions 11–12 · September 29 & October 1, 2026**

Every code cell from the [lecture note](https://harunpirim.github.io/IMSE774-NeuralNetworks/notes/06-fitting-models.html), in the order it appears
there. The prose, figures, and exercises live on that page; this notebook is
for running and changing the code.

Run a cell with **Shift+Enter**. Run the setup cell first — the later cells
depend on the names it defines.

Source: [https://github.com/harunpirim/IMSE774-NeuralNetworks](https://github.com/harunpirim/IMSE774-NeuralNetworks)


## Setup


`setup`


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.func import vmap, grad, hessian

torch.manual_seed(774)
np.random.seed(774)

plt.rcParams.update({
    "figure.figsize": (7, 4),
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.frameon": False,
})

NDSU_GREEN = "#005643"
NDSU_GOLD  = "#FFC72C"
ACCENT     = "#C1121F"
STEEL      = "#1864AB"

## Linear regression example


`fig-gd-linear`


In [ ]:
rng = np.random.default_rng(774)
x_lin = np.sort(rng.uniform(0, 2, 12))
y_lin = 0.85 + 0.45 * x_lin + rng.normal(0, 0.12, 12)
X_lin = np.column_stack([np.ones_like(x_lin), x_lin])

def loss_lin(phi):                                      # equation (6.5)
    return np.sum((X_lin @ phi - y_lin) ** 2)

def grad_lin(phi):                                      # equations (6.6) and (6.7)
    r = X_lin @ phi - y_lin
    return np.array([2 * r.sum(), 2 * (x_lin * r).sum()])

def line_search(f, phi, d, a_max=2.0, tol=1e-6):
    """Bracketing search (figure 6.10) for the step a along -d that minimizes f.
    Golden-section spacing lets each iteration reuse one interior point."""
    g = (np.sqrt(5) - 1) / 2
    a, b = 0.0, a_max
    c, e = b - g * (b - a), a + g * (b - a)
    while b - a > tol:
        if f(phi - c * d) < f(phi - e * d):             # minimum is not in [e, b]
            b, e = e, c
            c = b - g * (b - a)
        else:                                           # minimum is not in [a, c]
            a, c = c, e
            e = a + g * (b - a)
    return 0.5 * (a + b)

phi = np.array([1.7, -0.75])
path = [phi]
for _ in range(8):
    d = grad_lin(phi)                                   # step 1, equation (6.2)
    phi = phi - line_search(loss_lin, phi, d) * d       # step 2, equation (6.3)
    path.append(phi)
path = np.array(path)
phi_star = np.linalg.solve(X_lin.T @ X_lin, X_lin.T @ y_lin)

g0, g1 = np.meshgrid(np.linspace(0, 2, 300), np.linspace(-1, 1, 300))
L_surf = ((g0[..., None] + g1[..., None] * x_lin - y_lin) ** 2).sum(-1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
ax = axes[0]
ax.contourf(g0, g1, np.log10(L_surf), 30, cmap="copper")
ax.contour(g0, g1, np.log10(L_surf), 15, colors="white", linewidths=0.4, alpha=0.4)
ax.plot(path[:, 0], path[:, 1], "-o", color=NDSU_GOLD, ms=4, lw=1.4)
for k in range(5):
    ax.annotate(str(k), path[k], textcoords="offset points", xytext=(6, 4), color="white", fontsize=8)
ax.plot(*phi_star, "*", color="white", ms=13)
ax.set_xlabel(r"intercept $\phi_0$"); ax.set_ylabel(r"slope $\phi_1$"); ax.set_title(r"$\log_{10} L[\phi]$ and the iterates")
ax.grid(False)
ax = axes[1]
xs = np.linspace(0, 2, 50)
for k, p in enumerate(path[:5]):
    ax.plot(xs, p[0] + p[1] * xs, color=NDSU_GREEN, alpha=0.2 + 0.8 * k / 4, lw=2, label=f"iterate {k}" if k in (0, 4) else None)
ax.scatter(x_lin, y_lin, color=ACCENT, s=28, zorder=3)
ax.set_xlabel("$x$"); ax.set_ylabel("$y$"); ax.set_ylim(0, 2); ax.legend(fontsize=8)
ax.set_title("the model at each iterate")
plt.tight_layout()
plt.show()

print(f"{'iterate':>8}" + "".join(f"{k:>8}" for k in range(len(path))))
print(f"{'loss':>8}" + "".join(f"{loss_lin(p):>8.3f}" for p in path))
print(f"\nclosed-form optimum (normal equations): loss {loss_lin(phi_star):.4f}")

`fig-learning-rate`


In [ ]:
H_lin = 2 * X_lin.T @ X_lin                             # the Hessian of equation (6.5)
lam = np.linalg.eigvalsh(H_lin)
alpha_crit = 2 / lam[-1]

plt.figure(figsize=(6.4, 3.6))
colors = [STEEL, NDSU_GREEN, NDSU_GOLD, ACCENT]
for frac, c in zip((0.1, 0.5, 0.95, 1.05), colors):
    phi, hist = np.array([1.7, -0.75]), [loss_lin(np.array([1.7, -0.75]))]
    for _ in range(60):
        phi = phi - frac * alpha_crit * grad_lin(phi)
        hist.append(loss_lin(phi))
    plt.semilogy(hist, color=c, lw=2, label=rf"$\alpha = {frac}\times 2/\lambda_{{\max}}$")
plt.axhline(loss_lin(phi_star), color="gray", ls="--", lw=1)
plt.ylim(0.08, 1e3); plt.xlabel("iteration"); plt.ylabel("loss  (log scale)")
plt.legend(fontsize=8); plt.tight_layout(); plt.show()

print(f"Hessian eigenvalues: {lam.round(3)}   condition number {lam[-1] / lam[0]:.1f}")
print(f"stability limit 2 / lambda_max = {alpha_crit:.4f}")

## Non-Convex Loss Surfaces


`fig-gabor-critical`


In [ ]:
def gabor(x, phi):                                      # equation (6.8), batched over phi
    z = phi[..., 0:1] + 0.06 * phi[..., 1:2] * x
    return torch.sin(z) * torch.exp(-z ** 2 / 32.0)

x_g = torch.tensor(rng.uniform(-15, 15, 28))
y_g = gabor(x_g, torch.tensor([0.0, 16.6], dtype=torch.float64)) + torch.tensor(rng.normal(0, 0.15, 28))
I_g = len(x_g)

def L_w(phi, w):
    """Equation (6.9) with a 0/1 weight per example: all ones is the full loss,
    and a batch mask gives the loss of one batch (equation 6.10)."""
    return torch.sum(w * (gabor(x_g, phi) - y_g) ** 2)

ALL = torch.ones(I_g, dtype=torch.float64)
L_full = lambda phi: L_w(phi, ALL)
grad_L = vmap(grad(L_full))                             # gradients at many phi at once
hess_L = vmap(hessian(L_full))                          # 2 x 2 Hessians at many phi at once

p0g = torch.linspace(-10, 10, 301, dtype=torch.float64)
p1g = torch.linspace(2.5, 22.5, 301, dtype=torch.float64)
P0, P1 = torch.meshgrid(p0g, p1g, indexing="xy")
L_grid = ((gabor(x_g[None, :], torch.stack([P0.ravel(), P1.ravel()], 1)) - y_g) ** 2).sum(1).reshape(P0.shape)

starts = torch.stack(torch.meshgrid(torch.linspace(-10, 10, 41, dtype=torch.float64),
                                    torch.linspace(2.5, 22.5, 41, dtype=torch.float64),
                                    indexing="xy"), -1).reshape(-1, 2)
p = starts.clone()
for _ in range(60):                                     # Newton: phi <- phi - H^{-1} dL/dphi
    step = torch.linalg.solve(hess_L(p), grad_L(p).unsqueeze(-1)).squeeze(-1)
    p = p - step.clamp(-1.0, 1.0)                       # clamp keeps early steps from leaping off the map

inside = (p[:, 0].abs() <= 10) & (p[:, 1] >= 2.5) & (p[:, 1] <= 22.5)
p = p[(grad_L(p).norm(dim=1) < 1e-8) & inside]
crit = []
for q in p:                                             # merge duplicates found from different starts
    if all((q - c).norm() > 1e-3 for c in crit):
        crit.append(q)
crit = torch.stack(crit)
ev = torch.linalg.eigvalsh(hess_L(crit))
kind = np.where((ev > 0).all(1), "minimum", np.where((ev < 0).all(1), "maximum", "saddle"))
L_crit = torch.stack([L_full(c) for c in crit]).numpy()
is_min = kind == "minimum"
order = np.argsort(L_crit[is_min])
minima, L_minima = crit[is_min][order], L_crit[is_min][order]
phi_global = minima[0]

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4), gridspec_kw={"width_ratios": [1, 1.15]})
xs = torch.linspace(-15, 15, 400, dtype=torch.float64)
axes[0].scatter(x_g, y_g, color=ACCENT, s=26, zorder=3, label="training data")
axes[0].plot(xs, gabor(xs, phi_global).squeeze(), color=NDSU_GREEN, lw=2, label="model at global minimum")
axes[0].set_xlabel("$x$"); axes[0].set_ylabel("$y$"); axes[0].set_ylim(-1, 1); axes[0].legend(fontsize=8, loc="lower left")
ax = axes[1]
ax.contourf(P0, P1, torch.log(L_grid), 40, cmap="copper")
ax.contour(P0, P1, torch.log(L_grid), 18, colors="white", linewidths=0.3, alpha=0.35)
style = {"minimum": dict(marker="o", c="#7FDBDA", s=34, edgecolors="k", linewidths=0.5),
         "maximum": dict(marker="^", c="white", s=30, edgecolors="k", linewidths=0.4),
         "saddle":  dict(marker="x", c="white", s=30, linewidths=1.3)}
for k, st in style.items():
    pts = crit[kind == k]
    ax.scatter(pts[:, 0], pts[:, 1], label=f"{k} ({len(pts)})", **st)
ax.plot(*phi_global, "*", color=NDSU_GOLD, ms=15, mec="k", mew=0.5)
ax.set_xlim(-10, 10); ax.set_ylim(2.5, 22.5); ax.grid(False)
ax.set_xlabel(r"$\phi_0$"); ax.set_ylabel(r"$\phi_1$"); ax.set_title(r"$\log L[\phi]$ and its critical points")
ax.legend(fontsize=7.5, loc="upper left", framealpha=0.85, frameon=True)
plt.tight_layout()
plt.show()

print(f"critical points in the window: {is_min.sum()} minima, {(kind == 'maximum').sum()} maxima, "
      f"{(kind == 'saddle').sum()} saddle points")
print(f"global minimum: phi = [{phi_global[0]:.3f}, {phi_global[1]:.3f}], loss {L_minima[0]:.3f}"
      f"   (data generated from [0, 16.6])")
print(f"losses at the other minima: {np.round(L_minima[1:], 2)}")

## Local minima


`fig-multistart`


In [ ]:
def classify(p_end):
    """0 = global minimum, 1 = other local minimum, 2 = plateau, 3 = still moving."""
    L_end = torch.stack([L_full(q) for q in p_end])
    g_end = grad_L(p_end).norm(dim=1)
    at_global = (p_end - phi_global).norm(dim=1) < 0.1
    near_min = torch.stack([(p_end - m).norm(dim=1) < 0.1 for m in minima]).any(0)
    plateau = L_end > 0.95 * (y_g ** 2).sum()
    out = np.full(len(p_end), 3)
    out[plateau.numpy()] = 2
    out[(near_min & ~at_global).numpy()] = 1
    out[at_global.numpy()] = 0
    return out

p = starts.clone()
for _ in range(3000):
    p = p - 0.004 * grad_L(p)                           # equation (6.3), alpha = 0.004
gd_outcome = classify(p)

from matplotlib.colors import ListedColormap
OUTCOME_CMAP = ListedColormap([NDSU_GOLD, NDSU_GREEN, "#BBBBBB", STEEL])
OUTCOMES = ["global minimum", "other local minimum", "plateau", "still moving"]

fig, ax = plt.subplots(figsize=(5.4, 4.6))
ax.imshow(gd_outcome.reshape(41, 41), origin="lower", extent=(-10.25, 10.25, 2.25, 22.75),
          cmap=OUTCOME_CMAP, vmin=-0.5, vmax=3.5, aspect="auto", interpolation="nearest")
ax.scatter(minima[:, 0], minima[:, 1], marker="x", c="white", s=28, linewidths=1.3)
ax.set_xlabel(r"starting $\phi_0$"); ax.set_ylabel(r"starting $\phi_1$"); ax.grid(False)
ax.set_title("gradient descent: final destination")
plt.tight_layout(); plt.show()

for k, name in enumerate(OUTCOMES):
    print(f"{name:<22}{np.mean(gd_outcome == k):>7.1%}")

## Saddle points


`fig-saddle`


In [ ]:
def L_toy(p):  return p[0] ** 4 / 4 - p[0] ** 2 / 2 + p[1] ** 2 / 2
def g_toy(p):  return np.array([p[0] ** 3 - p[0], p[1]])

p = np.array([1e-7, 1.5]); gnorm, loss_toy = [], []
for _ in range(300):
    gnorm.append(np.linalg.norm(g_toy(p))); loss_toy.append(L_toy(p))
    p = p - 0.1 * g_toy(p)
gnorm, loss_toy = np.array(gnorm), np.array(loss_toy)
stop = int(np.argmax(gnorm < 1e-3))

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.3))
axes[0].semilogy(gnorm, color=NDSU_GREEN, lw=2)
axes[0].axhline(1e-3, color="gray", ls=":", lw=1.2)
axes[0].axvline(stop, color=ACCENT, lw=1, ls="--")
axes[0].set_xlabel("iteration"); axes[0].set_ylabel(r"$\|\partial L / \partial \phi\|$"); axes[0].set_title("gradient norm")
axes[1].plot(loss_toy, color=NDSU_GREEN, lw=2)
axes[1].axhline(0, color="gray", ls=":", lw=1); axes[1].axhline(-0.25, color="gray", ls="--", lw=1)
axes[1].axvline(stop, color=ACCENT, lw=1, ls="--")
axes[1].text(stop + 4, 0.6, "stopping rule\nfires here", color=ACCENT, fontsize=8)
axes[1].set_xlabel("iteration"); axes[1].set_ylabel("loss"); axes[1].set_title("loss"); axes[1].set_ylim(-0.35, 1.2)
plt.tight_layout(); plt.show()

print(f"the gradient norm first drops below 1e-3 at iteration {stop}: phi = {p.round(3)} at the end, but "
      f"at iteration {stop} the loss was {loss_toy[stop]:.2e} (saddle value 0)")
print(f"smallest gradient norm on the way past the saddle: {gnorm[40:150].min():.1e}")
print(f"final loss: {loss_toy[-1]:.4f}   (true minimum -0.25)")

## Batches and epochs


`fig-batch-gradients`


In [ ]:
phi_pt = torch.tensor([2.5, 9.0], dtype=torch.float64)
g_full = grad(L_full)(phi_pt)
gen = torch.Generator().manual_seed(774)

def batch_grads(B, n=4000):
    """Gradients of n random batches of size B at phi_pt, each the sum in equation (6.10)."""
    idx = torch.argsort(torch.rand(n, I_g, generator=gen), dim=1)[:, :B]
    W = torch.zeros(n, I_g, dtype=torch.float64).scatter_(1, idx, 1.0)
    return vmap(grad(L_w), in_dims=(None, 0))(phi_pt, W)

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.9))
gb3 = batch_grads(3, 300) * I_g / 3
uphill = (gb3 @ g_full) < 0
ax = axes[0]
for gvec, up in zip(gb3, uphill):
    ax.arrow(0, 0, -gvec[0].item(), -gvec[1].item(), color=ACCENT if up else STEEL,
             alpha=0.35, width=0.004, head_width=0.06, length_includes_head=True)
ax.arrow(0, 0, -g_full[0].item(), -g_full[1].item(), color=NDSU_GOLD, width=0.03,
         head_width=0.25, length_includes_head=True, zorder=5)
ax.set_aspect("equal"); ax.set_xlim(-2, 9); ax.set_ylim(-4, 4)
ax.set_xlabel(r"change in $\phi_0$"); ax.set_ylabel(r"change in $\phi_1$")
ax.set_title(f"batches of 3: {uphill.float().mean():.0%} of steps go uphill")

rows = []
for B in (1, 2, 3, 5, 7, 14, 28):
    gb = batch_grads(B)
    cos = (gb @ g_full) / (gb.norm(dim=1) * g_full.norm())
    rows.append((B, ((gb @ g_full) < 0).float().mean().item(), cos.median().item(),
                 (gb.mean(0) * I_g / B - g_full).norm().item() / g_full.norm().item()))
rows = np.array(rows)
ax = axes[1]
ax.plot(rows[:, 0], rows[:, 1], "o-", color=ACCENT, lw=2, label="fraction of uphill steps")
ax.plot(rows[:, 0], rows[:, 2], "s-", color=STEEL, lw=2, label="median cosine with full gradient")
ax.set_xscale("log"); ax.set_xticks(rows[:, 0]); ax.set_xticklabels([int(b) for b in rows[:, 0]])
ax.set_xlabel("batch size $B$  (of $I = 28$)"); ax.set_ylim(-0.02, 1.05); ax.legend(fontsize=8)
ax.set_title("noise shrinks as the batch grows")
plt.tight_layout(); plt.show()

print(f"{'B':>4}{'uphill':>9}{'median cos':>12}{'|mean(I/B * batch grad) - full| / |full|':>44}")
for B, up, c, err in rows:
    print(f"{int(B):>4}{up:>9.1%}{c:>12.3f}{err:>44.3f}")

## Properties of stochastic gradient descent


`fig-sgd-vs-gd`


In [ ]:
B_sgd, epochs, lr0 = 7, 300, 0.2
steps_per_epoch = I_g // B_sgd

def lr_at(epoch):                                       # halve every 75 epochs
    return lr0 * 0.5 ** (epoch // 75)

gen = torch.Generator().manual_seed(774)
p_sgd, p_gd = starts.clone(), starts.clone()
N = len(starts)
for ep in range(epochs):
    perm = torch.argsort(torch.rand(N, I_g, generator=gen), dim=1)     # a fresh shuffle per run per epoch
    for k in range(steps_per_epoch):
        W = torch.zeros(N, I_g, dtype=torch.float64).scatter_(1, perm[:, k * B_sgd:(k + 1) * B_sgd], 1.0)
        p_sgd = p_sgd - lr_at(ep) * vmap(grad(L_w))(p_sgd, W)            # equation (6.10)
        p_gd  = p_gd  - lr_at(ep) * B_sgd / I_g * grad_L(p_gd)           # same expected step, no noise
sgd_outcome, gdm_outcome = classify(p_sgd), classify(p_gd)

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.4))
for ax, out, title in [(axes[0], gdm_outcome, "full batch, matched step"), (axes[1], sgd_outcome, "SGD, batches of 7")]:
    ax.imshow(out.reshape(41, 41), origin="lower", extent=(-10.25, 10.25, 2.25, 22.75),
              cmap=OUTCOME_CMAP, vmin=-0.5, vmax=3.5, aspect="auto", interpolation="nearest")
    ax.scatter(minima[:, 0], minima[:, 1], marker="x", c="white", s=24, linewidths=1.2)
    ax.set_xlabel(r"starting $\phi_0$"); ax.grid(False); ax.set_title(title)
axes[0].set_ylabel(r"starting $\phi_1$")
plt.tight_layout(); plt.show()

print(f"{'outcome':<22}{'GD, fixed small step':>22}{'GD, matched step':>18}{'SGD':>8}")
for k, name in enumerate(OUTCOMES):
    print(f"{name:<22}{np.mean(gd_outcome == k):>22.1%}{np.mean(gdm_outcome == k):>18.1%}{np.mean(sgd_outcome == k):>8.1%}")

## Learning rate schedules


`fig-lr-schedule`


In [ ]:
x_std = (x_lin - x_lin.mean()) / x_lin.std()
X_std = np.column_stack([np.ones(12), x_std])
phi_hat = np.linalg.solve(X_std.T @ X_std, X_std.T @ y_lin)
L_hat = np.sum((X_std @ phi_hat - y_lin) ** 2)

def sgd_linear(lr_fn, epochs=60, B=2, seed=0):
    r = np.random.default_rng(seed)
    phi, excess = np.array([-1.0, 1.5]), []
    for ep in range(epochs):
        perm = r.permutation(12)                        # without replacement, one epoch
        for k in range(12 // B):
            b = perm[k * B:(k + 1) * B]
            phi = phi - lr_fn(ep) * 2 * X_std[b].T @ (X_std[b] @ phi - y_lin[b])
        excess.append(np.sum((X_std @ phi - y_lin) ** 2) - L_hat)
    return np.array(excess)

schedules = {"constant, $\\alpha = 0.2$":                 (lambda e: 0.2,                   ACCENT,     "-"),
             "constant, $\\alpha = 0.01$":                (lambda e: 0.01,                  STEEL,      "-"),
             "step decay: $0.2$, halved every 5 epochs":  (lambda e: 0.2 * 0.5 ** (e // 5),  NDSU_GREEN, "-"),
             "step decay: $0.2$, halved every 10 epochs": (lambda e: 0.2 * 0.5 ** (e // 10), NDSU_GREEN, "--")}

plt.figure(figsize=(6.6, 3.6))
summary = {}
for name, (fn, c, ls) in schedules.items():
    runs = np.array([sgd_linear(fn, seed=s) for s in range(20)])
    med = np.median(runs, axis=0); summary[name] = med
    plt.semilogy(np.arange(1, 61), med, ls, color=c, lw=2 if ls == "-" else 1.4, label=name)
plt.xlabel("epoch"); plt.ylabel(r"excess loss $L[\phi] - L[\hat\phi]$")
plt.legend(fontsize=8); plt.tight_layout(); plt.show()

small = summary["constant, $\\alpha = 0.01$"]
print(f"{'schedule':<44}{'epoch 5':>11}{'epoch 20':>11}{'epoch 60':>11}{'epochs behind alpha = 0.01':>29}")
for name, med in summary.items():
    print(f"{name.replace('$', '').replace(chr(92), ''):<44}{med[4]:>11.2e}{med[19]:>11.2e}{med[-1]:>11.2e}"
          f"{int(np.sum(med > small)):>29}")

## Momentum


`fig-momentum`


In [ ]:
A_q = np.diag([0.1, 10.0])
p_start = np.array([-0.9, -0.6])

def run_momentum(alpha, beta, nesterov=False, n=3000, tol=1e-6):
    """Equation (6.11), or (6.12) when nesterov=True, on the bowl; full-batch gradients."""
    phi, m, path = p_start.copy(), np.zeros(2), [p_start.copy()]
    for t in range(n):
        g = A_q @ (phi - alpha * m) if nesterov else A_q @ phi
        m = beta * m + (1 - beta) * g
        phi = phi - alpha * m
        path.append(phi.copy())
        if not np.all(np.isfinite(phi)) or np.abs(phi).max() > 1e6:
            return None, np.array(path)
        if np.linalg.norm(phi) < tol:
            return t + 1, np.array(path)
    return None, np.array(path)

alphas = np.geomspace(0.01, 5, 60)
betas = [0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 0.99]
best = {}
for name, nest, bset in [("gradient descent", False, [0.0]), ("momentum", False, betas),
                         ("Nesterov momentum", True, betas)]:
    trials = [(run_momentum(a, b, nest)[0] or 10 ** 9, a, b) for a in alphas for b in bset]
    best[name] = min(trials)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.9))
gx, gy = np.meshgrid(np.linspace(-1.05, 0.15, 200), np.linspace(-0.85, 1.15, 200))
axes[0].contour(gx, gy, 0.5 * (0.1 * gx ** 2 + 10 * gy ** 2), 18, colors="gray", linewidths=0.5, alpha=0.6)
for (name, (n, a, b)), c, lw in zip(best.items(), [ACCENT, NDSU_GREEN, STEEL], [0.8, 1.3, 1.3]):
    _, path = run_momentum(a, b, name.startswith("Nesterov"))
    axes[0].plot(path[:41, 0], path[:41, 1], "-o", color=c, ms=2.5, lw=lw, label=name)
    axes[1].semilogy(np.linalg.norm(path, axis=1), color=c, lw=2, label=f"{name}: {n} steps")
axes[0].plot(0, 0, "*", color=NDSU_GOLD, ms=13, mec="k", mew=0.5, zorder=5)
axes[0].set_xlim(-1.05, 0.15); axes[0].set_ylim(-0.85, 1.15)
axes[0].set_xlabel(r"$\phi_0$  (shallow direction)"); axes[0].set_ylabel(r"$\phi_1$  (steep direction)")
axes[0].legend(fontsize=8, loc="upper right"); axes[0].set_title("first 40 steps"); axes[0].grid(False)
axes[1].set_xlim(0, 800); axes[1].set_ylim(1e-6, 2)
axes[1].set_xlabel("iteration"); axes[1].set_ylabel(r"$\|\phi - \hat\phi\|$"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f"{'method':<20}{'alpha':>8}{'beta':>7}{'steps to 1e-6':>15}")
for name, (n, a, b) in best.items():
    print(f"{name:<20}{a:>8.3f}{b:>7.2f}{n:>15}")
print(f"\nstability limit of plain gradient descent, 2 / lambda_max = {2 / 10.0:.3f}")

## Adam


`fig-adam`


In [ ]:
A_a = np.diag([0.5, 10.0])
grad_a = lambda p: A_a @ p
p_start_a = np.array([-0.7, -0.9])

def optimize(update, n=100):
    phi, state, path = p_start_a.copy(), {}, [p_start_a.copy()]
    for t in range(n):
        phi = update(phi, grad_a(phi), t, state)
        path.append(phi.copy())
    return np.array(path)

def gd_step(alpha):
    return lambda phi, g, t, s: phi - alpha * g                              # equation (6.3)

def normalized_step(alpha, eps=1e-8):
    return lambda phi, g, t, s: phi - alpha * g / (np.sqrt(g ** 2) + eps)    # equations (6.13)-(6.14)

def adam_step(alpha, beta=0.9, gamma=0.99, eps=1e-8):
    def step(phi, g, t, s):
        s["m"] = beta * s.get("m", 0.0) + (1 - beta) * g                     # equation (6.15)
        s["v"] = gamma * s.get("v", 0.0) + (1 - gamma) * g ** 2
        m_hat = s["m"] / (1 - beta ** (t + 1))                               # equation (6.16)
        v_hat = s["v"] / (1 - gamma ** (t + 1))
        return phi - alpha * m_hat / (np.sqrt(v_hat) + eps)                  # equation (6.17)
    return step

panels = [("a) gradient descent, $\\alpha = 0.05$", gd_step(0.05)),
          ("b) gradient descent, $\\alpha = 0.195$", gd_step(0.195)),
          ("c) normalized gradients, $\\alpha = 0.05$", normalized_step(0.05)),
          ("d) Adam, $\\alpha = 0.05,\\ \\beta = 0.9,\\ \\gamma = 0.99$", adam_step(0.05))]

gx, gy = np.meshgrid(np.linspace(-1, 1, 200), np.linspace(-1, 1, 200))
L_bowl = 0.5 * (0.5 * gx ** 2 + 10 * gy ** 2)
fig, axes = plt.subplots(2, 2, figsize=(9.6, 7.6))
final = {}
for ax, (title, upd) in zip(axes.ravel(), panels):
    path = optimize(upd)
    final[title] = np.linalg.norm(path[[20, 50, 100]], axis=1)
    ax.contourf(gx, gy, np.log(L_bowl + 1e-3), 30, cmap="copper")
    ax.plot(path[:, 0], path[:, 1], "-o", color="#9EE3DF", ms=2.6, lw=0.9)
    ax.plot(*path[0], "o", color=NDSU_GREEN, ms=6)
    ax.set_title(title, fontsize=9.5); ax.set_xlim(-1, 1); ax.set_ylim(-1, 1); ax.grid(False)
    ax.set_xlabel(r"$\phi_0$"); ax.set_ylabel(r"$\phi_1$")
plt.tight_layout(); plt.show()

print(f"{'method':<52}{'|phi| after 20':>15}{'50':>11}{'100':>11}")
for title, d in final.items():
    print(f"{title.replace('$', '').replace(chr(92), ''):<52}{d[0]:>15.2e}{d[1]:>11.2e}{d[2]:>11.2e}")

`adam-checks`


In [ ]:
print("bias correction, equation (6.16), with PyTorch's defaults beta = 0.9, gamma = 0.999")
print("a constant gradient g = 1 gives these running averages before and after correction:\n")
print(f"{'t':>4}{'m_t':>9}{'m~_t':>8}{'v_t':>10}{'v~_t':>8}{'update without correction':>28}")
for t in range(1, 6):
    m_t, v_t = 1 - 0.9 ** t, 1 - 0.999 ** t                   # closed form of the recursion for g = 1
    print(f"{t:>4}{m_t:>9.4f}{1.0:>8.1f}{v_t:>10.4f}{1.0:>8.1f}{m_t / np.sqrt(v_t):>28.3f} x alpha")

torch.manual_seed(774)
Q = torch.randn(5, 5, dtype=torch.float64); Q = Q @ Q.T + 0.1 * torch.eye(5, dtype=torch.float64)
c = torch.randn(5, dtype=torch.float64)
quad = lambda w: 0.5 * w @ Q @ w - c @ w                     # a 5-parameter convex test loss
w_star = torch.linalg.solve(Q, c)

w_torch = torch.zeros(5, dtype=torch.float64, requires_grad=True)
opt = torch.optim.Adam([w_torch], lr=0.05, betas=(0.9, 0.99), eps=1e-8)
w_ours, s = np.zeros(5), {}
step = adam_step(0.05, beta=0.9, gamma=0.99)
gap = 0.0
for t in range(200):
    opt.zero_grad(); quad(w_torch).backward(); opt.step()
    w_ours = step(w_ours, (Q.numpy() @ w_ours - c.numpy()), t, s)
    gap = max(gap, np.abs(w_ours - w_torch.detach().numpy()).max())
print(f"\nour equations (6.15)-(6.17) vs torch.optim.Adam over 200 steps: max difference {gap:.1e}")

print("\nmultiply the same loss by k, keep every hyperparameter fixed, run 300 steps:")
print(f"{'k':>8}{'SGD (lr 0.15): |w - w*|':>28}{'Adam (lr 0.05): |w - w*|':>28}")
for k in (1.0, 1000.0):
    dist = []
    for make in (lambda p: torch.optim.SGD(p, lr=0.15), lambda p: torch.optim.Adam(p, lr=0.05)):
        w = torch.zeros(5, dtype=torch.float64, requires_grad=True)
        o = make([w])
        for _ in range(300):
            o.zero_grad(); (k * quad(w)).backward(); o.step()
        dist.append((w.detach() - w_star).norm().item())
    print(f"{k:>8.0f}{dist[0]:>28.2e}{dist[1]:>28.2e}")

## Optimizers in PyTorch


`momentum-convention`


In [ ]:
alpha, beta = 0.1, 0.9
w_t = torch.zeros(5, dtype=torch.float64, requires_grad=True)
opt = torch.optim.SGD([w_t], lr=alpha * (1 - beta), momentum=beta)   # PyTorch's convention
w, m, gap = np.zeros(5), np.zeros(5), 0.0
for _ in range(200):
    opt.zero_grad(); quad(w_t).backward(); opt.step()
    g = Q.numpy() @ w - c.numpy()
    m = beta * m + (1 - beta) * g                                       # Prince, equation (6.11)
    w = w - alpha * m
    gap = max(gap, np.abs(w - w_t.detach().numpy()).max())
print(f"Prince alpha = {alpha}, beta = {beta}  <=>  torch.optim.SGD(lr = {alpha * (1 - beta):.3f}, momentum = {beta})")
print(f"max difference over 200 steps: {gap:.1e}")
print(f"\nso torch.optim.SGD(lr = 0.05, momentum = 0.9) is Prince's alpha = {0.05 / (1 - 0.9):.2f}:")
print("adding momentum=0.9 to an SGD run multiplies its effective step size by ten.")

`fig-optimizers`


In [ ]:
rng_n = np.random.default_rng(774)
x_n = rng_n.uniform(-1, 1, 256)
y_n = np.sin(3 * x_n) + 0.3 * x_n ** 2 + rng_n.normal(0, 0.1, 256)
X_n = torch.tensor(x_n, dtype=torch.float32)[:, None]
Y_n = torch.tensor(y_n, dtype=torch.float32)[:, None]

OPTIMIZERS = {"SGD (lr 0.05)":                  lambda p: torch.optim.SGD(p, lr=0.05),
              "SGD + momentum (lr 0.05, 0.9)":  lambda p: torch.optim.SGD(p, lr=0.05, momentum=0.9),
              "Adam (lr 0.003)":                lambda p: torch.optim.Adam(p, lr=0.003)}

def train(make_opt, seed, epochs=150, B=16):
    torch.manual_seed(seed)
    net = nn.Sequential(nn.Linear(1, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 1))
    opt = make_opt(net.parameters())
    gen = torch.Generator().manual_seed(seed)
    history = []
    for _ in range(epochs):
        perm = torch.randperm(256, generator=gen)
        for k in range(256 // B):
            b = perm[k * B:(k + 1) * B]
            opt.zero_grad()
            loss = ((net(X_n[b]) - Y_n[b]) ** 2).mean()
            loss.backward()
            opt.step()
        with torch.no_grad():
            history.append(((net(X_n) - Y_n) ** 2).mean().item())
    return np.array(history)

plt.figure(figsize=(6.8, 3.7))
table = {}
for (name, make), c in zip(OPTIMIZERS.items(), [STEEL, NDSU_GREEN, ACCENT]):
    runs = np.array([train(make, s) for s in range(3)])
    table[name] = np.median(runs, axis=0)
    for k, h in enumerate(runs):
        plt.semilogy(np.arange(1, 151), h, color=c, lw=1.5, alpha=0.85, label=name if k == 0 else None)
plt.axhline(0.1 ** 2, color="gray", ls="--", lw=1)
plt.xlabel("epoch"); plt.ylabel("training MSE (log scale)"); plt.legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f"{'optimizer':<32}{'epoch 10':>10}{'epoch 50':>10}{'epoch 150':>11}   (median of 3 runs)")
for name, h in table.items():
    print(f"{name:<32}{h[9]:>10.4f}{h[49]:>10.4f}{h[-1]:>11.4f}")